# Autonomous Agent Prediction: A-to-Z Guide

Author: Sidhaarth Shree

Kaggle's **Autonomous Agent Prediction (Beta)** challenge asks you to submit an *agent* — not a static model — that can autonomously explore a dataset, train a model, and submit predictions on its own. Your submission is a small filesystem of config and prompt files (an `agent.yaml` spec) that a harness runs against 16 hidden tabular datasets.

This notebook is a full walkthrough for anyone joining the challenge: it covers exploring the data, building a schema-agnostic baseline, scaffolding a valid agent submission, and validating that submission before you upload it.

> **Note on scope:** This notebook explains everything the *code itself* does, in detail. It does not restate Kaggle's official rules, prize structure, or deadlines — for those, always check the [competition Rules and Overview tabs](https://www.kaggle.com/competitions/autonomous-agent-prediction-beta) directly, since that's the authoritative and most current source.

## How the challenge is structured (from the data itself)

- You get **16 training datasets** (`train_01` … `train_16`), each with its own `train.csv`, `test.csv`, and `solution.csv`.
- Every dataset uses the same generic schema: a `row_id` column, a binary `target` column, and an arbitrary mix of numeric/categorical feature columns — this is what lets one schema-agnostic pipeline work across all 16.
- `solution.csv` contains a `Usage` column marking rows as **Public** or **Private** — this is the standard Kaggle leaderboard split: your public score is computed on a subset visible during the competition, and the final ranking uses the private subset, revealed only at the end.
- Your deliverable isn't predictions directly — it's an **agent** (a folder of YAML/markdown files) that a harness executes. The harness gives your agent tools (`run_command`, `write_file`, `submit_predictions`, etc.) and your agent decides how to use them across all 16 datasets.

## Notebook roadmap

1. **Blocks 1–4** — Explore the data: locate it, summarize every dataset defensively, chart comparisons, and check the public/private split.
2. **Blocks 5–6** — Build and benchmark a schema-agnostic baseline model across all 16 datasets.
3. **Blocks 7–9** — Scaffold a valid `agent.yaml` submission, then validate it structurally (including a deliberately broken example to prove the validator works).
4. **Blocks 10–12** — Test whether feature engineering and per-dataset tuning improve on the baseline (and honestly diagnose which datasets have a real data-scarcity ceiling), then write that logic into the actual submission skill and re-validate.

## Block 1 — Setup & path detection

Kaggle mounts competition data under `/kaggle/input/...`, but the exact path differs depending on whether you're in a notebook attached to the competition directly or one where the dataset was added manually. This block tries the two most likely fixed paths first, and falls back to a `glob` search for a `train_01` folder if neither exists — so the notebook doesn't break just because of where the data happens to be mounted. It fails loudly (`FileNotFoundError`) rather than silently, so you don't waste time debugging a `NoneType` path later.

In [ ]:
import glob
import os
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

CANDIDATE_ROOTS = [
    "/kaggle/input/competitions/autonomous-agent-prediction-beta/data",
    "/kaggle/input/autonomous-agent-prediction-beta/data",
]

DATA_ROOT = next((p for p in CANDIDATE_ROOTS if os.path.isdir(p)), None)
if DATA_ROOT is None:
    hits = glob.glob("/kaggle/input/**/train_01", recursive=True)
    if hits:
        DATA_ROOT = os.path.dirname(hits[0])

if DATA_ROOT is None:
    raise FileNotFoundError(
        "Couldn't locate the data/ folder automatically — check the Data tab "
        "for the exact path and set DATA_ROOT manually."
    )

dataset_dirs = sorted(glob.glob(os.path.join(DATA_ROOT, "train_*")))
print(f"Found {len(dataset_dirs)} training datasets under:\n  {DATA_ROOT}")

## Block 2 — Per-dataset EDA loop (defensive, one file at a time)

With 16 datasets of unknown size and schema, looping over all of them in one shot is risky — one oversized or malformed file can crash the whole notebook before you see any results. This block wraps each dataset's summary in a `try/except`, prints progress as it goes (`flush=True` so output shows immediately even in a slow Kaggle kernel), and calls `gc.collect()` after each iteration to free memory before moving to the next dataset.

For each dataset it computes: row/feature counts, the numeric/categorical split, average missing-value fraction, the positive class rate (how imbalanced `target` is), and the single feature most correlated with `target` — a fast signal for how learnable each dataset looks.

In [ ]:
import random

MAX_ROWS_FOR_EDA = 150_000  # row cap per dataset — keeps memory bounded, prevents OOM kernel restarts

def _count_rows(path):
    with open(path, "rb") as f:
        return sum(1 for _ in f) - 1  # minus header

def _read_capped(path, n_rows_total, max_rows):
    """Read the whole file if small; otherwise a random row sample so stats stay representative."""
    if n_rows_total <= max_rows:
        df = pd.read_csv(path)
    else:
        keep_frac = max_rows / n_rows_total
        rng = random.Random(0)
        df = pd.read_csv(
            path,
            skiprows=lambda i: i > 0 and rng.random() > keep_frac,  # i==0 is the header, always keep
        )
    # downcast dtypes to cut memory further
    for col in df.select_dtypes(include=["float64"]).columns:
        df[col] = df[col].astype("float32")
    for col in df.select_dtypes(include=["int64"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="integer")
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].astype("category")
    return df


def summarize_dataset(train_path, test_path):
    n_train_total = _count_rows(train_path)
    n_test_total = _count_rows(test_path) if os.path.exists(test_path) else np.nan

    train = _read_capped(train_path, n_train_total, MAX_ROWS_FOR_EDA)

    feature_cols = [c for c in train.columns if c not in ("row_id", "target")]
    num_cols = train[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in feature_cols if c not in num_cols]

    missing_frac = train[feature_cols].isna().mean().mean()
    pos_rate = train["target"].mean() if "target" in train.columns else np.nan

    if num_cols:
        max_abs_corr = train[num_cols].corrwith(train["target"]).abs().max()
    else:
        max_abs_corr = np.nan

    stats = {
        "n_train": n_train_total,          # true row count (not the sampled size)
        "n_test": n_test_total,
        "n_features": len(feature_cols),
        "n_numeric": len(num_cols),
        "n_categorical": len(cat_cols),
        "missing_frac": round(missing_frac, 3),
        "positive_rate": round(pos_rate, 3) if pd.notna(pos_rate) else np.nan,
        "max_abs_corr": round(max_abs_corr, 3) if pd.notna(max_abs_corr) else np.nan,
    }
    del train
    return stats


rows = []
for d in dataset_dirs:
    name = os.path.basename(d)
    train_path = os.path.join(d, "train.csv")
    test_path = os.path.join(d, "test.csv")

    size_mb = os.path.getsize(train_path) / (1024 * 1024) if os.path.exists(train_path) else float("nan")
    print(f"[{name}] train.csv = {size_mb:.2f} MB", flush=True)

    try:
        stats = summarize_dataset(train_path, test_path)
        rows.append({"dataset": name, **stats})
        print(f"[{name}] OK -> shape=({stats['n_train']}, {stats['n_features']})", flush=True)
    except Exception as e:
        print(f"[{name}] FAILED: {type(e).__name__}: {e}", flush=True)
    finally:
        gc.collect()

summary = pd.DataFrame(rows).sort_values("dataset").reset_index(drop=True)
summary

## Block 3 — Comparison charts

Three side-by-side bar charts turn the `summary` table into something you can scan at a glance: positive rate per dataset (with a 0.5 reference line so imbalance jumps out), feature count per dataset, and mean missing-fraction per dataset. Useful for spotting outlier datasets — e.g. one that's heavily imbalanced or has unusually sparse features — before you sink time into modeling them.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(summary["dataset"], summary["positive_rate"], color="#2E86AB")
axes[0].set_title("Positive rate by dataset")
axes[0].axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
axes[0].tick_params(axis="x", rotation=90)

axes[1].bar(summary["dataset"], summary["n_features"], color="#A23B72")
axes[1].set_title("Feature count by dataset")
axes[1].tick_params(axis="x", rotation=90)

axes[2].bar(summary["dataset"], summary["missing_frac"], color="#F18F01")
axes[2].set_title("Mean missing fraction by dataset")
axes[2].tick_params(axis="x", rotation=90)

plt.tight_layout()
plt.show()

## Block 4 — Inner public/private split check (`Usage` column)

`solution.csv` (available to you because this is training data, not the hidden test set) contains a `Usage` column that mirrors how Kaggle splits every leaderboard: rows marked **Public** feed the public leaderboard score you see during the competition; rows marked **Private** are held back and only used for final scoring after the deadline. Checking the proportion of each per dataset tells you roughly how much of your score is "locked" until the end — useful context for not over-trusting your public leaderboard rank.

In [ ]:
usage_rows = []
for d in dataset_dirs:
    name = os.path.basename(d)
    sol_path = os.path.join(d, "solution.csv")
    if not os.path.exists(sol_path):
        continue

    try:
        # peek at the header only (near-zero memory) before loading any data
        header_cols = pd.read_csv(sol_path, nrows=0).columns
        if "Usage" not in header_cols:
            continue

        # read only the Usage column, not the whole file
        usage_col = pd.read_csv(sol_path, usecols=["Usage"])["Usage"].astype("category")

        counts = usage_col.value_counts(normalize=True)
        usage_rows.append({"dataset": name, **counts.to_dict()})
        del usage_col
        print(f"[{name}] OK", flush=True)
    except Exception as e:
        print(f"[{name}] FAILED: {type(e).__name__}: {e}", flush=True)
    finally:
        gc.collect()

usage_df = pd.DataFrame(usage_rows).sort_values("dataset").reset_index(drop=True)
usage_df

## Block 5 — Schema-agnostic baseline pipeline (functions)

Since all 16 datasets share the same *shape* of schema (id, target, arbitrary features) but not the same *columns*, this block builds one reusable `sklearn` pipeline rather than 16 bespoke ones:

- **Numeric features** → median imputation, then standard scaling.
- **Categorical features** → constant imputation with a `"__missing__"` placeholder, then one-hot encoding (`handle_unknown="ignore"` so unseen categories at inference don't crash it).
- **Model** → `HistGradientBoostingClassifier`, a solid, fast default for mixed tabular data that natively tolerates the missing values imputation didn't catch.

`cv_auc()` wraps this in 5-fold stratified cross-validation scored on ROC-AUC, and caps any dataset over 200k rows with a fixed-seed downsample (`MAX_ROWS_FOR_CV`) purely so CV time stays predictable — this is a speed/safety guard, not a modeling recommendation for your final agent.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

MAX_ROWS_FOR_CV = 200_000  # safety cap so one oversized dataset can't stall/crash the loop


def build_pipeline(train_df, target_col="target"):
    feature_cols = [c for c in train_df.columns if c not in ("row_id", target_col)]
    num_cols = train_df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in feature_cols if c not in num_cols]

    numeric_pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="__missing__")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    pre = ColumnTransformer(
        [("num", numeric_pipe, num_cols), ("cat", categorical_pipe, cat_cols)],
        sparse_threshold=0,  # force dense output even when the OHE step produces sparse data
    )
    clf = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.06, random_state=0)
    return Pipeline([("pre", pre), ("clf", clf)]), feature_cols


def cv_auc(train_df, target_col="target", n_splits=5):
    if len(train_df) > MAX_ROWS_FOR_CV:
        train_df = train_df.sample(MAX_ROWS_FOR_CV, random_state=0)
        print(f"  (downsampled to {MAX_ROWS_FOR_CV} rows for CV speed/safety)", flush=True)
    pipe, feature_cols = build_pipeline(train_df, target_col)
    X, y = train_df[feature_cols], train_df[target_col]
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)
    scores = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc")
    return scores.mean(), scores.std()

## Block 6 — Baseline results across all 16 datasets (defensive)

Runs `cv_auc()` over every dataset with the same defensive pattern as Block 2 — `try/except` per dataset, progress printed immediately, `gc.collect()` between iterations — so one bad dataset produces a `NaN` row instead of killing the whole run. The resulting `baseline_df` (mean ± std ROC-AUC per dataset) is your reference point: any modeling improvement your agent makes should beat these numbers, and any dataset where the baseline is unexpectedly weak is worth extra feature-engineering attention.

In [ ]:
baseline_rows = []
for d in dataset_dirs:
    name = os.path.basename(d)
    print(f"[{name}] loading...", flush=True)
    try:
        train = pd.read_csv(os.path.join(d, "train.csv"))
        mean_auc, std_auc = cv_auc(train)
        baseline_rows.append({"dataset": name, "cv_auc_mean": round(mean_auc, 4), "cv_auc_std": round(std_auc, 4)})
        print(f"[{name}] cv_auc={mean_auc:.4f} +/- {std_auc:.4f}", flush=True)
        del train
    except Exception as e:
        print(f"[{name}] FAILED: {type(e).__name__}: {e}", flush=True)
        baseline_rows.append({"dataset": name, "cv_auc_mean": np.nan, "cv_auc_std": np.nan})
    finally:
        gc.collect()

baseline_df = pd.DataFrame(baseline_rows).sort_values("dataset").reset_index(drop=True)
baseline_df

## Block 7 — Demo submission scaffold: the `agent.yaml` spec, field by field

This is where the challenge diverges from a normal Kaggle submission: instead of a `submission.csv`, you submit a small folder the harness executes. Block 7 generates a minimal but complete example, `demo_submission/`:

| File | Purpose |
|---|---|
| `agent.yaml` | The **root agent config** — every submission needs exactly one of these at its root. |
| `agent.yaml: name` | Identifier for this agent (used in harness logs). |
| `agent.yaml: model` | Which model powers this agent's reasoning (e.g. `gemini-3.5-flash`). |
| `agent.yaml: instruction` | The system prompt. Can be inline text, or `!include path/to/file.md` to pull it from another file — keeps `agent.yaml` itself short and prompts easy to iterate on separately. |
| `agent.yaml: tools` | The list of harness capabilities this agent may call — e.g. `run_command`, `write_file`, `submit_predictions`, `select_submission`, `get_status`. |
| `agent.yaml: tools[].agent_tool` | A **sub-agent** entry: instead of a plain tool name, points to another `.yaml` config (via `config_path`), letting one agent delegate a sub-task to another agent with its own model/instructions/tools. Here, the root agent delegates EDA to a `data_analyst` sub-agent. |
| `agent.yaml: skills` | Paths to skill folders, each requiring its own `SKILL.md` manifest — reusable capabilities (like the `feature-engineer` skill here) the agent can invoke. |
| `agent.yaml: generate_content_config` | Sampling controls for this agent's model calls — `temperature`, `max_output_tokens`, and an optional `thinking_config` (`thinking_budget`, `include_thoughts`). The real spec expects this key on every agent config, root or sub-agent; Block 8's validator now checks for it. |

The `!include` syntax and the `agent_tool` delegation pattern are what make this a *hierarchical* agent spec rather than a flat one — Block 8's validator has to walk this hierarchy correctly to check it.

In [ ]:
os.makedirs("demo_submission/prompts", exist_ok=True)
os.makedirs("demo_submission/tools", exist_ok=True)
os.makedirs("demo_submission/skills/feature-engineer", exist_ok=True)

with open("demo_submission/agent.yaml", "w") as f:
    f.write("""name: ml_agent
model: gemini-3.5-flash
instruction: !include prompts/system.md
tools:
  - run_command
  - write_file
  - submit_predictions
  - select_submission
  - get_status
  - agent_tool:
      config_path: tools/data_analyst.yaml
skills:
  - skills/feature-engineer
generate_content_config:
  temperature: 0.12
  max_output_tokens: 8192
  thinking_config:
    thinking_budget: 2048
    include_thoughts: true
""")

with open("demo_submission/prompts/system.md", "w") as f:
    f.write("You are an ML agent. Delegate EDA to data_analyst, then model and submit.")

with open("demo_submission/tools/data_analyst.yaml", "w") as f:
    f.write("""name: data_analyst
model: gemini-3.1-flash-lite
instruction: !include ../prompts/data_analyst.md
tools:
  - run_command
  - write_file
generate_content_config:
  temperature: 0.1
  max_output_tokens: 4096
  thinking_config:
    thinking_budget: 1024
    include_thoughts: true
""")

with open("demo_submission/prompts/data_analyst.md", "w") as f:
    f.write("You perform EDA only. Do not build models.")

with open("demo_submission/skills/feature-engineer/SKILL.md", "w") as f:
    f.write("---\nname: feature-engineer\ndescription: automated feature generation\n---\n# Feature Engineer\n")

print("demo_submission/ created (agent.yaml + data_analyst.yaml both declare generate_content_config)")


## Block 8 — Structural validator

Before you ever upload a submission, you want to know it's structurally sound — missing files or a typo'd tool name will otherwise only surface as an opaque harness failure later. This validator:

- Parses `!include` as a **custom YAML tag** (`IncludeTag`) rather than letting it error out, so it can inspect *what's being included* without needing a full templating engine.
- **Recursively walks `agent_tool` delegations** (Block 7's `data_analyst` sub-agent), tracking visited configs in `_seen` to avoid infinite loops if configs ever reference each other circularly.
- Checks every plain-string tool against an `ALLOWED_TOOLS` allowlist — catching typos or tools the harness doesn't actually support.
- **Guards against path traversal**: every resolved path (`config_relpath`, `!include` targets) is normalized and checked that it still starts with the submission root — so a config can't reference files outside `demo_submission/` (accidentally or otherwise).
- Confirms every required key (`name`, `model`, `instruction`, `tools`) is present, every `!include` target file exists, and every referenced skill folder has a `SKILL.md`.
- **Checks every config in the hierarchy for `generate_content_config`** (root agent *and* every `agent_tool` sub-agent) and, if present, that it carries `temperature` / `max_output_tokens` and a well-formed `thinking_config`. The original version of this validator didn't check for this key at all — it's part of the real spec, so a submission missing it would still pass structurally while being incomplete.

Run this on your real submission before uploading — an empty issues list is your go/no-go signal.

In [ ]:
import yaml

ALLOWED_TOOLS = {
    "run_command", "write_file", "edit_file", "submit_predictions",
    "select_submission", "get_status", "run_skill_script", "load_skill_resource",
}


class IncludeTag:
    def __init__(self, path):
        self.path = path


def _include_constructor(loader, node):
    return IncludeTag(loader.construct_scalar(node))


yaml.SafeLoader.add_constructor("!include", _include_constructor)


def validate_agent_config(root, config_relpath="agent.yaml", _seen=None):
    issues = []
    _seen = _seen if _seen is not None else set()
    root_abs = os.path.normpath(root)
    config_path = os.path.normpath(os.path.join(root_abs, config_relpath))

    if not config_path.startswith(root_abs):
        issues.append(f"path traversal outside submission root: {config_relpath}")
        return issues
    if config_path in _seen:
        return issues
    _seen.add(config_path)
    if not os.path.exists(config_path):
        issues.append(f"missing file: {config_relpath}")
        return issues

    with open(config_path) as f:
        cfg = yaml.safe_load(f)

    config_dir_rel = os.path.dirname(config_relpath)

    for required in ("name", "model", "instruction", "tools"):
        if required not in cfg:
            issues.append(f"{config_relpath}: missing required key '{required}'")

    # generate_content_config: the real agent.yaml spec expects this key on every config
    # (root and sub-agents alike) — it controls sampling temperature, max output length, and
    # thinking budget. This was never checked before; a submission could pass structurally
    # while silently missing it.
    gcc = cfg.get("generate_content_config")
    if gcc is None:
        issues.append(f"{config_relpath}: missing 'generate_content_config' (expected by the agent.yaml spec)")
    elif not isinstance(gcc, dict):
        issues.append(f"{config_relpath}: 'generate_content_config' must be a mapping")
    else:
        for key in ("temperature", "max_output_tokens"):
            if key not in gcc:
                issues.append(f"{config_relpath}: generate_content_config missing '{key}'")
        thinking_cfg = gcc.get("thinking_config")
        if thinking_cfg is not None and "thinking_budget" not in thinking_cfg:
            issues.append(f"{config_relpath}: generate_content_config.thinking_config missing 'thinking_budget'")

    for tool in cfg.get("tools", []):
        if isinstance(tool, str):
            if tool not in ALLOWED_TOOLS:
                issues.append(f"{config_relpath}: '{tool}' is not an allowed harness tool")
        elif isinstance(tool, dict) and "agent_tool" in tool:
            sub_cfg = tool["agent_tool"].get("config_path")
            if not sub_cfg:
                issues.append(f"{config_relpath}: agent_tool missing config_path")
            else:
                sub_relpath = os.path.normpath(os.path.join(config_dir_rel, sub_cfg))
                issues.extend(validate_agent_config(root, sub_relpath, _seen))
        else:
            issues.append(f"{config_relpath}: unrecognized tool entry {tool!r}")

    instruction = cfg.get("instruction")
    if isinstance(instruction, IncludeTag):
        included_relpath = os.path.normpath(os.path.join(config_dir_rel, instruction.path))
        included_abspath = os.path.normpath(os.path.join(root_abs, included_relpath))
        if not included_abspath.startswith(root_abs):
            issues.append(f"{config_relpath}: !include for 'instruction' escapes submission root")
        elif not os.path.exists(included_abspath):
            issues.append(f"{config_relpath}: !include target not found: {instruction.path}")

    for skill_dir in cfg.get("skills", []):
        skill_md = os.path.join(root_abs, skill_dir, "SKILL.md")
        if not os.path.exists(skill_md):
            issues.append(f"skill '{skill_dir}' is missing a SKILL.md manifest")

    return issues


issues = validate_agent_config("demo_submission")
print("Issues found:", issues if issues else "none — structure looks valid ✅")


## Block 9 — Broken-config demo (proves the validator catches problems)

A second, deliberately broken config: `!include ../../outside/system.md` tries to escape the submission root, `delete_everything` isn't in `ALLOWED_TOOLS`, and `skills/does-not-exist` has no `SKILL.md`. It still ships a valid `generate_content_config` block on purpose, so the validator's new check doesn't add a fourth, unrelated issue to the list — running it should surface exactly the three deliberate breaks. If it does, you can trust the validator on your real submission too.

In [ ]:
with open("demo_submission/agent_broken.yaml", "w") as f:
    f.write("""name: ml_agent
model: gemini-3.5-flash
instruction: !include ../../outside/system.md
tools:
  - run_command
  - delete_everything
skills:
  - skills/does-not-exist
generate_content_config:
  temperature: 0.12
  max_output_tokens: 8192
  thinking_config:
    thinking_budget: 2048
    include_thoughts: true
""")

broken_issues = validate_agent_config("demo_submission", "agent_broken.yaml")
for issue in broken_issues:
    print("-", issue)


## Block 10 — Feature engineering + hyperparameter search (diagnostic pass)

The baseline in Block 6 uses one fixed model config for all 16 datasets. This block tests
whether targeted feature engineering and per-dataset tuning can lift scores further: row-wise
numeric aggregates, interaction terms between the top-correlated features, K-fold (leakage-safe)
target encoding for categoricals, and a `RandomizedSearchCV` over `HistGradientBoostingClassifier`
hyperparameters per dataset.

This is a **diagnostic run**, not the final submission logic — its job is to tell you honestly
where your ceiling is before you commit to a modeling strategy. On this data it surfaced an
important finding: `train_05`, `train_09`, and `train_13` have only 500–1,109 training rows and
don't respond to added complexity — confirmed by comparing against plain logistic regression,
which matched or beat the complex pipeline on all three. That's a genuine data-scarcity ceiling,
not a modeling gap.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, KFold
from scipy.stats import randint, uniform
from itertools import combinations

def add_row_meta_features(df, num_cols):
    if num_cols:
        df["_row_mean"] = df[num_cols].mean(axis=1)
        df["_row_std"] = df[num_cols].std(axis=1)
        df["_row_max"] = df[num_cols].max(axis=1)
        df["_row_min"] = df[num_cols].min(axis=1)
        df["_row_null_count"] = df[num_cols].isna().sum(axis=1)
    return df


def add_interaction_features(df, num_cols, target_col, top_k=4):
    """Pairwise products of the top_k numeric columns most correlated with target —
    catches non-linear/interaction signal a tree model can still miss on small data."""
    if len(num_cols) < 2 or target_col not in df.columns:
        return df
    corrs = df[num_cols].corrwith(df[target_col]).abs().sort_values(ascending=False)
    top_cols = corrs.head(top_k).index.tolist()
    for a, b in combinations(top_cols, 2):
        df[f"_int_{a}_{b}"] = df[a] * df[b]
    return df


def kfold_target_encode(df, cat_cols, target_col, n_splits=5, seed=0):
    """Out-of-fold mean-target encoding: each row's encoded value is computed from
    OTHER folds only, so it doesn't leak the row's own label into its own feature."""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    global_mean = df[target_col].mean()
    for c in cat_cols:
        oof = pd.Series(index=df.index, dtype=float)
        for tr_idx, val_idx in kf.split(df):
            means = df.iloc[tr_idx].groupby(c)[target_col].mean()
            oof.iloc[val_idx] = df.iloc[val_idx][c].map(means)
        df[c + "_te"] = oof.fillna(global_mean)
    return df


def apply_target_encoding_to_test(train_df, test_df, cat_cols, target_col):
    """For a held-out test set, encode using the FULL train set's means (no leakage risk
    here since test has no label to leak)."""
    global_mean = train_df[target_col].mean()
    for c in cat_cols:
        means = train_df.groupby(c)[target_col].mean()
        test_df[c + "_te"] = test_df[c].map(means).fillna(global_mean)
    return test_df


def build_pipeline_v3(train_df, target_col="target"):
    feature_cols = [c for c in train_df.columns if c not in ("row_id", target_col)]
    num_cols = train_df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in feature_cols if c not in num_cols]

    numeric_pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="__missing__")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    pre = ColumnTransformer(
        [("num", numeric_pipe, num_cols), ("cat", categorical_pipe, cat_cols)],
        sparse_threshold=0,
    )
    clf = HistGradientBoostingClassifier(random_state=0)
    return Pipeline([("pre", pre), ("clf", clf)]), feature_cols


def cv_auc_tuned_v2(train_df, target_col="target", n_splits=5, n_iter=25):
    if len(train_df) > MAX_ROWS_FOR_CV:
        train_df = train_df.sample(MAX_ROWS_FOR_CV, random_state=0)

    train_df = train_df.copy()
    orig_feature_cols = [c for c in train_df.columns if c not in ("row_id", target_col)]
    num_cols = train_df[orig_feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in orig_feature_cols if c not in num_cols]

    train_df = add_row_meta_features(train_df, num_cols)
    train_df = add_interaction_features(train_df, num_cols, target_col)
    if cat_cols:
        train_df = kfold_target_encode(train_df, cat_cols, target_col)
        train_df = train_df.drop(columns=cat_cols)  # keep the _te version, drop raw high-cardinality cats

    pipe, feature_cols = build_pipeline_v3(train_df, target_col)
    X, y = train_df[feature_cols], train_df[target_col]

    param_dist = {
        "clf__max_iter": randint(150, 500),
        "clf__learning_rate": uniform(0.01, 0.2),
        "clf__max_leaf_nodes": randint(10, 80),
        "clf__max_depth": randint(2, 10),
        "clf__min_samples_leaf": randint(5, 50),
        "clf__l2_regularization": uniform(0.0, 2.0),
    }
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)
    search = RandomizedSearchCV(
        pipe, param_dist, n_iter=n_iter, cv=cv, scoring="roc_auc",
        random_state=0, n_jobs=-1,
    )
    search.fit(X, y)
    return search.best_score_, search.cv_results_["std_test_score"][search.best_index_], search.best_params_


improved_rows = []
for d in dataset_dirs:
    name = os.path.basename(d)
    print(f"[{name}] tuning...", flush=True)
    try:
        train = pd.read_csv(os.path.join(d, "train.csv"))
        mean_auc, std_auc, best_params = cv_auc_tuned_v2(train)
        improved_rows.append({"dataset": name, "cv_auc_mean": round(mean_auc, 4), "cv_auc_std": round(std_auc, 4)})
        print(f"[{name}] cv_auc={mean_auc:.4f} +/- {std_auc:.4f} | best={best_params}", flush=True)
        del train
    except Exception as e:
        print(f"[{name}] FAILED: {type(e).__name__}: {e}", flush=True)
        improved_rows.append({"dataset": name, "cv_auc_mean": np.nan, "cv_auc_std": np.nan})
    finally:
        gc.collect()

improved_df = pd.DataFrame(improved_rows).sort_values("dataset").reset_index(drop=True)
comparison = baseline_df.merge(improved_df, on="dataset", suffixes=("_baseline", "_tuned"))
comparison["lift"] = comparison["cv_auc_mean_tuned"] - comparison["cv_auc_mean_baseline"]
print(f"\nMean baseline AUC: {comparison['cv_auc_mean_baseline'].mean():.4f}")
print(f"Mean tuned AUC:    {comparison['cv_auc_mean_tuned'].mean():.4f}")
comparison

## Block 11 — Writing the real `feature-engineer` skill (schema-agnostic + defensive)

This replaces the earlier one-line `system.md` stub and the fixed-schema skill script with the
real submission logic. Several rounds of testing against the actual harness surfaced fixes
beyond the original two gaps:

**`feature_engineer.py` is schema-agnostic, not fixed to `train.csv`/`test.csv`/`row_id`/`target`.**
It scans the task directory for CSVs, picks train/test/sample files by name pattern (falling back
to a file-size heuristic if names are unhelpful, while excluding solution/answer/ground-truth and
submission-like files from ever being picked), then infers the **ID column** and **target column**
from context rather than hardcoding names, and normalizes non-numeric binary labels (e.g.
`"Yes"`/`"No"`) to `{0, 1}`.

**It has a runtime and memory safety net.** A 500k-row dataset with a couple of higher-cardinality
categorical columns made the original unhardened version run past 4.5 minutes with no result —
`RandomizedSearchCV` had no cap at all. It now downsamples above `MAX_ROWS_FOR_FIT` rows before any
feature engineering or search, and scales `n_iter`/CV folds down for larger datasets and when the
time already spent is eating into `TIME_BUDGET_SECONDS`. The same 500k-row case now finishes in
well under a minute.

**It always writes a valid `submission.csv`, even on total failure.** If discovery, inference, or
modeling raises anywhere, `prior_fallback()` writes a constant-probability submission instead of
crashing with nothing submitted.

**Every file write strips a leading blank line (`.lstrip("\n")`).** Without it, `SKILL.md` doesn't
start with `---` as its very first byte, and the harness's frontmatter parser rejects it outright
with `SKILL.md must start with YAML frontmatter (---)` — a real error from an actual run, not a
hypothetical one.

**`system.md` assumes one dataset per session, not sixteen.** The 16 `train_01..train_16` sets are
local practice data only — a deployed session gets exactly one hidden dataset. A system prompt
written as a 16-dataset loop can leave an agent looking for datasets that don't exist and never
reach `submit_predictions`. It's rewritten around a single session, with the skill script and
`submit_predictions` called back-to-back immediately after `get_status` — no EDA delegation, no
planning, in between. EDA is still available, just demoted to an optional step *after* a baseline
submission is already locked in, so a stall or a thought loop can only cost you an improvement,
never the whole session. An explicit **Anti-Loop Rule** gives the agent an unambiguous default
action ("run the script" or "submit what you have") if it's unsure what to do next.

The engineered-feature logic itself (row-wise numeric aggregates, interaction terms, K-fold target
encoding) is the pipeline validated in Block 10, and still **branches on training-set size**:
datasets under `small_n_threshold` rows skip the complex feature engineering — shown in Block 10 to
overfit on tiny folds — in favor of a strongly regularized logistic regression.


In [ ]:
system_md_text = '''
# Mission
You are `ml_agent`, dropped into a sandboxed environment containing exactly one hidden
binary-classification dataset for this session. There is nothing to loop over and nothing to
plan at length -- your first priority is locking in one valid submission as fast as possible,
not designing the best possible approach before acting.

# Hard Rules
- Never read, open, copy, or otherwise use any solution, ground-truth, answer, or test-label file.
- Do not access the internet. Do not install or upgrade any package.
- Predictions must be probabilities in [0, 1] -- never hard 0/1 labels.
- If sample_submission.csv exists, preserve its exact columns, row order, and row count.
- Only call tools listed in agent.yaml.
- Calling submit_predictions at least once, successfully, before the session ends is mandatory
  -- a session that ends without it scores zero no matter how good the modeling was.

# Anti-Loop Rule
Do not spend more than one or two turns reasoning before you call a tool. If you notice yourself
re-explaining the plan, reconsidering the same decision, or narrating what you're about to do
instead of doing it, stop and immediately call the next tool in the Execution Plan below. When in
doubt about what to do next, the answer is: run the skill script if you haven't yet, or call
submit_predictions if a valid submission.csv already exists. Never end the session without having
called submit_predictions at least once.

# Execution Plan
1. Call get_status immediately to see remaining time/budget.
2. Immediately run the feature-engineer skill script -- do not inspect the data or plan first:
   python skills/feature-engineer/feature_engineer.py . submission.csv
   It auto-discovers files, infers ID/target, engineers features, fits a model, and always
   writes a valid submission.csv, with a fallback if anything fails. This step alone is enough
   to produce something submittable.
3. Call submit_predictions on submission.csv right away. This must happen before any EDA,
   analysis, or attempt to improve on it -- a locked-in baseline is the single most important
   outcome of this session, and every step after this one is optional.
4. Only after step 3 has succeeded, and only if meaningful budget remains, delegate one concise
   EDA pass to data_analyst to look for a specific, well-understood improvement.
5. If EDA surfaces a concrete change (not a vague hunch), make ONE low-risk edit, re-run the
   skill script, and call submit_predictions again only if local CV genuinely improves.
6. Call get_status, then select_submission with your best submission ID.
7. Stop as soon as budget is low, or after at most one improvement attempt -- do not keep
   iterating for its own sake.

# Modeling Priorities
- Assume ROC-AUC scoring. Preserve ranking quality; do not threshold predictions into labels.
- Prefer robust, generalizing models over anything that looks like leaderboard probing.
- Treat a suspiciously high local validation score as a leakage warning, not a win.

# If Something Goes Wrong
If the skill script errors out or is unavailable, fall back to the simplest thing that still
produces a valid file: a plain logistic regression, or -- if even that fails -- a constant
prior-probability submission.csv. A valid weak submission always beats a crash with nothing
submitted, and there is no human watching this session to notice and fix it for you.
'''

data_analyst_md_text = '''
You are a concise EDA subagent for a hidden binary-classification task.

Use run_command with small Python snippets only. Do not build models, do not submit, and never
open any solution or ground-truth file.

Report only the following:
1. Discovered files and likely train/test/sample paths (names may not match earlier datasets).
2. Train/test shapes and columns.
3. Target column source: target_col.txt hint, train-only column, or binary-column inference.
4. Target balance.
5. Numeric/categorical counts, high-cardinality columns, constants, and missingness.
6. Simple train/test drift warnings for shared numeric columns.
7. Leakage warnings and a short modeling recommendation.

Keep the report compact and actionable.
'''

skill_md_text = '''
---
name: feature-engineer
description: schema-agnostic binary-classification playbook -- discovers train/test/sample files, infers ID/target columns, engineers row-meta/interaction/target-encoded features, tunes HistGradientBoostingClassifier (or a regularized LogisticRegression on small datasets) within a wall-clock and row-count safety budget, and always writes a valid submission.csv with a fallback on failure
---
# Feature Engineer

Run via: `python feature_engineer.py <task_dir> <output.csv>`

`task_dir` is any directory containing that dataset's train/test/sample CSVs -- their exact
filenames don't need to match `train.csv`/`test.csv`/`sample_submission.csv`. The script:

- Discovers files by name pattern, with a file-size fallback if names are unhelpful. Solution,
  answer, ground-truth, and submission-like files are never picked as train/test.
- Infers the ID column (from `sample_submission.csv`'s first column, common ID-like names, or
  uniqueness in both train and test) and the target column (from an optional `target_col.txt`
  hint, a train-only column, or binary-ness), and normalizes non-numeric binary labels to {0, 1}.
- Datasets under 2,000 rows: strongly regularized logistic regression only (Block 10 showed
  complex feature engineering overfits on tiny folds).
- Larger datasets: row-wise numeric aggregates, pairwise interaction terms on the top-correlated
  numeric columns, K-fold (leakage-safe) target encoding for categoricals, and a
  RandomizedSearchCV-tuned HistGradientBoostingClassifier.
- Downsamples above MAX_ROWS_FOR_FIT rows and scales search effort down for larger datasets or
  when TIME_BUDGET_SECONDS is running out, so a large or high-cardinality hidden dataset can't
  consume the whole session before writing a submission.
- Always writes a valid `submission.csv`. If discovery, inference, or modeling fails anywhere,
  `prior_fallback()` writes a constant-probability submission instead of crashing with nothing
  submitted.
'''

feature_engineer_code = '''
import argparse
import glob
import os
import time
from itertools import combinations

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from scipy.stats import randint, uniform

# Hard runtime/size safety net. Without these, a large or high-cardinality hidden dataset
# can make RandomizedSearchCV run far longer than any autonomous session's time budget,
# eating the whole session before submission.csv is ever written.
TIME_BUDGET_SECONDS = 180   # soft wall-clock cap on the search phase
MAX_ROWS_FOR_FIT = 150_000  # downsample above this, same pattern as the notebook's own CV cap


# ---------- File discovery (schema-agnostic) ----------

def _is_bad_name(path):
    name = os.path.basename(path).lower()
    return any(x in name for x in ("solution", "answer", "truth", "ground"))


def discover_files(task_dir):
    # Find train/test/sample_submission CSVs by scanning task_dir — no fixed filenames assumed.
    csvs = [
        p for p in glob.glob(os.path.join(task_dir, "**", "*.csv"), recursive=True)
        if not _is_bad_name(p)
    ]

    def pick(include, exclude=()):
        for p in sorted(csvs, key=len):
            n = os.path.basename(p).lower()
            if any(k in n for k in include) and not any(e in n for e in exclude):
                return p
        return None

    train = pick(("train",), exclude=("sample", "submission"))
    test = pick(("test",), exclude=("sample", "submission"))
    sample = pick(("sample_submission", "samplesubmission", "sample_sub"))

    if train is None or test is None:
        remaining = [
            p for p in csvs
            if p not in (train, test, sample)
            and not any(x in os.path.basename(p).lower() for x in ("sample", "submission"))
        ]
        remaining.sort(key=os.path.getsize, reverse=True)
        if train is None and remaining:
            train = remaining.pop(0)
        if test is None and remaining:
            test = remaining.pop(0)

    if train is None or test is None:
        raise FileNotFoundError(f"could not locate train/test CSVs under {task_dir}; found {csvs[:10]}")

    return train, test, sample


def infer_id_column(train, test, sample):
    if sample is not None and len(sample.columns) >= 1 and sample.columns[0] in test.columns:
        return sample.columns[0]
    preferred = ("row_id", "id", "sample_id", "uid", "index")
    for c in test.columns:
        if str(c).lower() in preferred or str(c).lower().endswith("_id"):
            return c
    for c in test.columns:
        if c in train.columns and test[c].is_unique and train[c].is_unique:
            return c
    return None


def infer_target_column(train, test, sample, id_col, hint=None):
    if hint and hint in train.columns:
        return hint
    train_only = [c for c in train.columns if c not in test.columns and c != id_col]
    likely = ("target", "label", "class", "y", "outcome")
    for c in train_only:
        if str(c).lower() in likely:
            return c
    if len(train_only) == 1:
        return train_only[0]
    binary_cols = [
        c for c in train.columns
        if c != id_col and c not in test.columns
        and train[c].nunique(dropna=True) == 2
    ]
    if binary_cols:
        return binary_cols[0]
    if sample is not None and len(sample.columns) >= 2:
        for c in sample.columns[1:]:
            if c in train.columns:
                return c
    raise ValueError("could not infer target column")


def normalize_binary_target(train, target_col):
    # Some hidden datasets may encode the label as e.g. "Yes"/"No" instead of 0/1.
    # Map to {0, 1} so downstream CV scoring (roc_auc) and sklearn classifiers behave
    # consistently; the larger of the two values (numeric or lexicographic) becomes 1.
    y = train[target_col]
    if pd.api.types.is_numeric_dtype(y) and set(y.dropna().unique()) <= {0, 1}:
        return train
    uniques = list(pd.unique(y.dropna()))
    if len(uniques) != 2:
        return train  # not binary in the expected sense — leave as-is, let it fail loudly downstream
    try:
        positive = max(uniques)
    except TypeError:
        positive = sorted(uniques, key=str)[-1]
    train[target_col] = (y == positive).astype(int)
    return train


def read_target_hint(train_path):
    for cand in (
        os.path.join(os.path.dirname(train_path), "target_col.txt"),
        os.path.join(os.path.dirname(train_path), "..", "target_col.txt"),
    ):
        if os.path.isfile(cand):
            with open(cand) as f:
                txt = f.read().strip()
            if txt:
                return txt
    return None


# ---------- Feature engineering (same logic validated in Block 10) ----------

def add_row_meta_features(df, num_cols):
    if num_cols:
        df["_row_mean"] = df[num_cols].mean(axis=1)
        df["_row_max"] = df[num_cols].max(axis=1)
        df["_row_min"] = df[num_cols].min(axis=1)
        df["_row_null_count"] = df[num_cols].isna().sum(axis=1)
        if len(num_cols) >= 2:
            # std over a single column is NaN for every row (ddof=1 with n=1) — only
            # meaningful, and only added, once there are at least 2 numeric columns.
            df["_row_std"] = df[num_cols].std(axis=1)
    return df


def add_interaction_features(train, test, num_cols, target_col, top_k=4):
    if len(num_cols) < 2:
        return train, test
    corrs = train[num_cols].corrwith(train[target_col]).abs().sort_values(ascending=False)
    top_cols = corrs.head(top_k).index.tolist()
    for a, b in combinations(top_cols, 2):
        train[f"_int_{a}_{b}"] = train[a] * train[b]
        test[f"_int_{a}_{b}"] = test[a] * test[b]
    return train, test


def kfold_target_encode(train, cat_cols, target_col, n_splits=5, seed=0):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    global_mean = train[target_col].mean()
    for c in cat_cols:
        oof = pd.Series(index=train.index, dtype=float)
        for tr_idx, val_idx in kf.split(train):
            means = train.iloc[tr_idx].groupby(c)[target_col].mean()
            oof.iloc[val_idx] = train.iloc[val_idx][c].map(means)
        train[c + "_te"] = oof.fillna(global_mean)
    return train


def apply_target_encoding_to_test(train, test, cat_cols, target_col):
    global_mean = train[target_col].mean()
    for c in cat_cols:
        means = train.groupby(c)[target_col].mean()
        test[c + "_te"] = test[c].map(means).fillna(global_mean)
    return test


def build_pipeline(num_cols, cat_cols, clf):
    numeric_pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="__missing__")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    pre = ColumnTransformer([("num", numeric_pipe, num_cols), ("cat", categorical_pipe, cat_cols)], sparse_threshold=0)
    return Pipeline([("pre", pre), ("clf", clf)])


# ---------- Main pipeline ----------

def train_and_predict(task_dir, small_n_threshold=2000, n_iter=8):
    t0 = time.time()
    train_path, test_path, sample_path = discover_files(task_dir)
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    sample = pd.read_csv(sample_path) if sample_path else None

    id_col = infer_id_column(train, test, sample)
    hint = read_target_hint(train_path)
    target_col = infer_target_column(train, test, sample, id_col, hint)
    train = normalize_binary_target(train, target_col)

    if len(train) > MAX_ROWS_FOR_FIT:
        # Same safety cap as the notebook's own baseline CV (MAX_ROWS_FOR_CV) — without it,
        # a large hidden dataset can make the search below run far past the session's time
        # budget with nothing submitted.
        train = train.sample(MAX_ROWS_FOR_FIT, random_state=0).reset_index(drop=True)

    feature_cols = [c for c in train.columns if c not in (id_col, target_col) and c in test.columns]
    num_cols = train[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in feature_cols if c not in num_cols]

    if len(train) < small_n_threshold:
        pipe = build_pipeline(num_cols, cat_cols, LogisticRegression(max_iter=2000, C=0.1))
        X, y = train[feature_cols], train[target_col]
        n_splits = max(2, min(5, y.value_counts().min()))
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)
        cv_score = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc").mean()
        pipe.fit(X, y)
        preds = pipe.predict_proba(test[feature_cols])[:, 1]
    else:
        train = add_row_meta_features(train, num_cols)
        test = add_row_meta_features(test, num_cols)
        train, test = add_interaction_features(train, test, num_cols, target_col)
        if cat_cols:
            train = kfold_target_encode(train, cat_cols, target_col)
            test = apply_target_encoding_to_test(train, test, cat_cols, target_col)
            train = train.drop(columns=cat_cols)
            test = test.drop(columns=cat_cols)

        feature_cols = [c for c in train.columns if c not in (id_col, target_col) and c in test.columns]
        num_cols = train[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

        pipe = build_pipeline(num_cols, [], HistGradientBoostingClassifier(random_state=0))
        X, y = train[feature_cols], train[target_col]

        # Scale search effort to both dataset size and time already spent on discovery/feature
        # engineering above — large datasets and/or a search that's already running late both
        # get a cheaper search rather than an unbounded one.
        elapsed = time.time() - t0
        remaining = TIME_BUDGET_SECONDS - elapsed
        if remaining < 30:
            n_iter_eff, n_splits_eff = 1, 3
        elif len(train) > 80_000:
            n_iter_eff, n_splits_eff = min(n_iter, 3), 4   # moderate cut, not maximal
        else:
            n_iter_eff, n_splits_eff = n_iter, 5

        param_dist = {
            "clf__max_iter": randint(100, 400),
            "clf__learning_rate": uniform(0.02, 0.15),
            "clf__max_leaf_nodes": randint(15, 63),
            "clf__l2_regularization": uniform(0.0, 1.0),
        }
        cv = StratifiedKFold(n_splits=n_splits_eff, shuffle=True, random_state=0)
        search = RandomizedSearchCV(
            pipe, param_dist, n_iter=n_iter_eff, cv=cv, scoring="roc_auc", random_state=0, n_jobs=-1
        )
        search.fit(X, y)
        preds = search.best_estimator_.predict_proba(test[feature_cols])[:, 1]
        cv_score = search.best_score_

    ids = test[id_col] if id_col else pd.RangeIndex(len(test))
    out = pd.DataFrame({(id_col or "row_id"): ids, target_col: preds})
    return out, cv_score, target_col, id_col


def prior_fallback(task_dir, out_path):
    # Last-resort submission: always write something valid, even if everything above failed.
    try:
        train_path, test_path, sample_path = discover_files(task_dir)
        test = pd.read_csv(test_path)
        sample = pd.read_csv(sample_path) if sample_path else None
        try:
            train = pd.read_csv(train_path)
            id_col = infer_id_column(train, test, sample)
            hint = read_target_hint(train_path)
            target_col = infer_target_column(train, test, sample, id_col, hint)
            prior = float(np.clip(train[target_col].mean(), 1e-6, 1 - 1e-6))
        except Exception:
            id_col, target_col, prior = None, "target", 0.5
        if sample is not None:
            out = sample.copy()
            out[out.columns[-1]] = prior
        else:
            ids = test[id_col] if id_col and id_col in test.columns else range(len(test))
            out = pd.DataFrame({(id_col or "row_id"): ids, target_col: prior})
        out.to_csv(out_path, index=False)
        print(f"fallback: wrote prior-probability submission ({prior:.4f}) -> {out_path}")
    except Exception as e:
        pd.DataFrame({"row_id": [0], "target": [0.5]}).to_csv(out_path, index=False)
        print(f"fallback failed too ({type(e).__name__}: {e}); wrote a single-row placeholder -> {out_path}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("task_dir", nargs="?", default=".", help="directory containing this dataset's train/test/sample CSVs")
    parser.add_argument("out_path", nargs="?", default="submission.csv", help="where to write the predictions CSV")
    args = parser.parse_args()

    try:
        preds_df, cv_score, target_col, id_col = train_and_predict(args.task_dir)
        preds_df.to_csv(args.out_path, index=False)
        print(f"cv_auc={cv_score:.4f} target={target_col!r} id={id_col!r} -> wrote {args.out_path}")
    except Exception as e:
        print(f"pipeline failed ({type(e).__name__}: {e}); writing fallback submission")
        prior_fallback(args.task_dir, args.out_path)


if __name__ == "__main__":
    main()
'''

os.makedirs("demo_submission/skills/feature-engineer", exist_ok=True)
with open("demo_submission/skills/feature-engineer/feature_engineer.py", "w") as f:
    f.write(feature_engineer_code.lstrip("\n"))

with open("demo_submission/skills/feature-engineer/SKILL.md", "w") as f:
    f.write(skill_md_text.lstrip("\n"))

with open("demo_submission/prompts/system.md", "w") as f:
    f.write(system_md_text.lstrip("\n"))

with open("demo_submission/prompts/data_analyst.md", "w") as f:
    f.write(data_analyst_md_text.lstrip("\n"))

print("feature_engineer.py (schema-agnostic + runtime-safe + fallback) + defensive system.md + updated SKILL.md/data_analyst.md written")


## Block 12 — Re-validating the updated submission

Same validator from Block 8, run again now that `feature_engineer.py` and the prompts have
changed. An empty issues list here confirms the updated skill files and prompts didn't break
the submission's structure — including that `agent.yaml` and `tools/data_analyst.yaml` still both
carry a valid `generate_content_config` — your final go/no-go check before committing.

In [ ]:
issues = validate_agent_config("demo_submission")
print("Issues found:", issues if issues else "none — structure looks valid ✅")

## Block 13 — Packaging `submission.zip`

Everything so far has built and validated a **folder** (`demo_submission/`). Kaggle doesn't take
a folder or a notebook output — the deliverable is a single `submission.zip` with `agent.yaml`
at its root.

This zips `demo_submission/` into `submission.zip`, stripping `__pycache__`/`.pyc` files (a
build artifact, not part of the real submission) and excluding `agent_broken.yaml` (Block 9's
deliberately-broken teaching file). It then re-opens the zip itself and checks the same
required-files list the harness needs, straight from the archive — the folder can be valid while
the zip you actually upload is stale if you rebuilt something after the last time you zipped.


In [ ]:
import shutil
import zipfile

SUBMISSION_ROOT = "demo_submission"
EXCLUDE_NAMES = {"agent_broken.yaml"}  # Block 9's deliberately-broken demo -- not part of the real agent

for root, dirs, files in os.walk(SUBMISSION_ROOT):
    if "__pycache__" in dirs:
        shutil.rmtree(os.path.join(root, "__pycache__"))

if os.path.exists("submission.zip"):
    os.remove("submission.zip")

with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(SUBMISSION_ROOT):
        dirs[:] = [d for d in dirs if d != "__pycache__"]
        for fname in files:
            if fname in EXCLUDE_NAMES or fname.endswith(".pyc"):
                continue
            full_path = os.path.join(root, fname)
            arcname = os.path.relpath(full_path, SUBMISSION_ROOT)
            zf.write(full_path, arcname)

size_kb = os.path.getsize("submission.zip") / 1024
print(f"OK: submission.zip written ({size_kb:.1f} KB)")

required = [
    "agent.yaml",
    "prompts/system.md",
    "prompts/data_analyst.md",
    "tools/data_analyst.yaml",
    "skills/feature-engineer/SKILL.md",
    "skills/feature-engineer/feature_engineer.py",
]

with zipfile.ZipFile("submission.zip") as zf:
    names = sorted(zf.namelist())

missing = [f for f in required if f not in names]
junk = [f for f in names if "__pycache__" in f or f.endswith(".pyc")]

print("\nArchive contents:")
for n in names:
    print("  ", n)

assert not missing, f"Missing required files: {missing}"
assert not junk, f"Unexpected cache files in ZIP: {junk}"
print("\nOK: all required files present, no junk in the archive -- ready to upload.")


## Glossary

- **`row_id`** — unique row identifier column, excluded from features.
- **`target`** — the binary label being predicted; drives `positive_rate` and AUC scoring.
- **`Usage` (Public/Private)** — Kaggle's standard leaderboard split; public rows score you during the competition, private rows decide the final rank.
- **ROC-AUC** — the metric used for cross-validation here; threshold-independent ranking quality of predicted probabilities.
- **`agent.yaml`** — the root config of an agent submission: name, model, instruction, tools, skills.
- **`!include`** — custom YAML tag pointing to an external file (e.g. a prompt) instead of inlining its content.
- **`agent_tool`** — a tool entry that's actually a pointer to another agent config, enabling sub-agent delegation.
- **`SKILL.md`** — the manifest file every skill folder must contain to be considered valid.
- **`generate_content_config`** — the sampling/thinking block (`temperature`, `max_output_tokens`, `thinking_config`) every agent config in the hierarchy is expected to declare.
- **Schema-agnostic discovery** — inferring which files are train/test/sample and which columns are ID/target from context, instead of assuming fixed filenames or column names like `row_id`/`target`.
- **Fallback submission** — a constant-probability `submission.csv` written when the main modeling path fails for any reason, so the agent never submits nothing.
- **`submission.zip`** — the actual file uploaded to Kaggle; a zipped copy of the submission folder with `agent.yaml` at its root. Validating the folder isn't the same as validating the zip you upload.
- **Anti-Loop Rule** — an explicit instruction giving the agent an unambiguous default action when it's unsure what to do next, to prevent it from stalling in reasoning without calling a tool.
- **Path traversal** — a config referencing files outside its own submission root (e.g. via `../../`); the validator explicitly blocks this.

## Where to go next

- Official rules, deadlines, evaluation metric, and prize details: the [competition Rules/Overview tabs](https://www.kaggle.com/competitions/autonomous-agent-prediction-beta) — always the source of truth, since those can change.